# 01. Fundamentals

## Learning goals
1. Name the built-in types and say which are mutable.
2. Predict truthiness, slicing and comprehension results.
3. Handle errors with `try`, `except`, `else` and `finally`.
4. Prove a loop invariant and state its cost.

In [ ]:
import sys, pathlib
for _p in (pathlib.Path.cwd(), pathlib.Path.cwd().parent):
    if (_p / "src" / "understanding_python").exists():
        sys.path.insert(0, str(_p / "src"))
print("python", sys.version.split()[0])

## Types and mutability
Every value is an object with a type. `int`, `float`, `str`, `tuple` and `frozenset` are immutable.
`list`, `dict` and `set` are mutable. Immutable objects can act as dictionary keys because their hash never changes.

In [ ]:
immutable = [1, 2.5, "text", (1, 2), frozenset({1})]
mutable = [[1], {"a": 1}, {1, 2}]
for v in immutable:
    hash(v)  # works
for v in mutable:
    try:
        hash(v)
        raise AssertionError("mutable types must not hash")
    except TypeError:
        pass
t = (1, [2, 3])
t[1].append(4)          # the tuple holds a reference; the list can still change
assert t == (1, [2, 3, 4])
print("hashability checks passed")

## Truthiness, slicing and unpacking
Empty containers, zero and `None` are falsy. A slice `a[i:j:k]` copies. Extended unpacking uses `*`.

In [ ]:
assert not [] and not {} and not "" and not 0 and not None
assert [0] and "0"
a = list(range(10))
assert a[2:5] == [2, 3, 4] and a[::-1][0] == 9 and a[-3:] == [7, 8, 9]
first, *mid, last = a
assert (first, last, len(mid)) == (0, 9, 8)
b = a[:]
b[0] = 99
assert a[0] == 0       # the slice copied the list

## Comprehensions and generator expressions
A comprehension builds a whole container. A generator expression yields lazily. Both run in time linear in the input.

In [ ]:
squares = [x * x for x in range(10) if x % 2 == 0]
assert squares == [0, 4, 16, 36, 64]
lookup = {c: i for i, c in enumerate("abc")}
assert lookup == {"a": 0, "b": 1, "c": 2}
total = sum(x for x in range(1000))
assert total == 499500

## Exceptions
`else` runs when no exception fired. `finally` always runs.

In [ ]:
log = []
def parse(s):
    try:
        n = int(s)
    except ValueError:
        log.append("bad")
        return None
    else:
        log.append("ok")
        return n
    finally:
        log.append("done")
assert parse("7") == 7 and parse("x") is None
assert log == ["ok", "done", "bad", "done"]

## Complexity: sum of the first n integers
**Claim.** The loop below returns n(n+1)/2.

**Proof by invariant.** Before iteration i, `total` equals 1+2+...+(i-1) = (i-1)i/2.
Initially i=1 and total=0, which equals 0. If the invariant holds before iteration i, then after adding i
the total is (i-1)i/2 + i = i(i+1)/2, so it holds for i+1. At exit i=n+1, so total = n(n+1)/2. The loop runs n times, so the cost is O(n).
The closed form costs O(1).

In [ ]:
def sum_loop(n):
    total = 0
    for i in range(1, n + 1):
        total += i
    return total
for n in (0, 1, 10, 1000):
    assert sum_loop(n) == n * (n + 1) // 2

## Benchmark: loop against comprehension against closed form
Sizes and seeds stay fixed so the run takes under a second.

In [ ]:
from understanding_python.bench import best_time
N = 20_000
def with_loop():
    out = []
    for i in range(N):
        out.append(i * 2)
    return out
def with_comp():
    return [i * 2 for i in range(N)]
t_loop, t_comp = best_time(with_loop, 20), best_time(with_comp, 20)
t_form = best_time(lambda: N * (N + 1) // 2, 20)
print(f"loop {t_loop:.4f}s  comprehension {t_comp:.4f}s  closed form {t_form:.6f}s")
assert with_loop() == with_comp()
assert t_form < t_comp       # O(1) beats O(n)